In [70]:
import pandas as pd
import numpy as np
import glob
import os
import gc
import hashlib
import pyarrow
from functions import *

Load OMI data - read - concatenate

In [71]:
# Grab all CSV files
folder = "data_origin/omi_estimate"

all_files = glob.glob(os.path.join(folder, "*.csv"))

out_dir = "datasets/omi_estimate"
os.makedirs(out_dir, exist_ok=True)


# Concatenate all datasets into a single DataFrame
dfs = []

for i,f in enumerate(all_files):
    """
    Read every dataset in the folder as input.

    Extract year and semester from the file name.

    Select relevant columns.

    Return datasets with updated istat codes. 
    """
    # Extract filename without extension
    filename = os.path.splitext(os.path.basename(f))[0]
    
    # Extract semester code
    parts = filename.split("_")
    semester_code = parts[-2]  # second to last part
    year = semester_code[:4]
    sem = semester_code[4]
    semester = f"{year}_S{sem}"
    
    # Read CSV, skip first title line
    df = pd.read_csv(f, sep=';', skiprows=1)
    
    # Strip whitespace and remove BOM from column names
    df.columns = df.columns.str.strip().str.replace('\ufeff','')

    # Semester columns: year_semester -> year + S1/S2; semester -> 1/2
    df['year_semester'] = semester
    df['semester'] = sem
    df['year'] = year

    # Keep relevant columns
    columns = [
        'Comune_ISTAT', 'Comune_descrizione', 'year', 'year_semester', 'semester', 'Zona', 
        'Descr_Tipologia', 'Stato', 'Compr_min', 'Compr_max'
        ]

    df = df[columns]

    # Translate columns names
    column_renames = {
        'Comune_ISTAT' : 'mun_istat', 
        'Comune_descrizione' : 'mun_name', 
        'Zona' : 'zone',
        'Descr_Tipologia' : 'type',
        'Stato' : 'condition',
        'Compr_min' : 'buy_min',
        'Compr_max' : 'buy_max'
    }

    df = df.rename(columns=column_renames)

    # Translate values names
    df["type"] = df["type"].replace({
        "Abitazioni civili": "Residential housing",
        "Box": "Garages",
        "Ville e Villini": "Independent houses and villas",
        "Negozi": "Shops",
        "Abitazioni di tipo economico": "Lowcost housing",
        "Magazzini": "Warehouses",
        "Uffici": "Offices",
        "Laboratori": "Laboratories",
        "Capannoni tipici": "Typical industrial buildings",
        "Capannoni industriali": "Industrial buildings",
        "Autorimesse": "Garages",
        "Posti auto scoperti": "Uncovered parking spaces",
        "Posti auto coperti": "Covered parking spaces",
        "Centri commerciali": "Shopping centers",
        "Uffici strutturati": "Structured offices",
        "Abitazioni tipiche dei luoghi": "Typical local housing",
        "Abitazioni signorili": "Luxury housing",
        "Pensioni e assimilati": "Guesthouses and similar",
        "Fabbricati e locali per esercizi sportivi": "Sports facilities"
    })

    df["condition"] = df["condition"].replace({
        "NORMALE": "Normal",
        "OTTIMO": "Excellent",
        "SCADENTE": "Poor"
    })
    
    # Convert numeric columns to int
    numeric_cols = ['buy_min', 'buy_max']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')
        df[col] = df[col].astype('Int64')

    # Normalize municipality names
    df['mun_name'] = df['mun_name'].apply(normalize_name)

    # Extract ISTAT code from the last 6 digits of the 'mun_istat' column -
    # add missing 0s - transform to object (preserve leading 0s)
    df['mun_istat'] = df['mun_istat'].astype('Int64').astype('str').str[-6:]

    add_zeroes(df, ['mun_istat'], 6)

    df['mun_istat'] = df['mun_istat'].astype('object')

    dfs.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(dfs, ignore_index=True)

# Sort by mun_istat, year, semester, type
final_df = final_df.sort_values(by=['mun_istat', 'year', 'semester', 'type'])

Duplicated Istat codes

In [72]:
# Count the number of duplicate listings
duplicates = final_df.value_counts(subset=[
    'mun_istat', 'zone', 'year_semester', 'condition', 'type'
    ])

duplicates = duplicates[duplicates > 1]

print("Number of duplicate listings for the same semester:", duplicates.sum())

Number of duplicate listings for the same semester: 358685


In [73]:
# Delete duplicate listings for the same semester - keep the first occurrence
final_df = final_df.drop_duplicates(subset=[
    'mun_istat', 'zone', 'year_semester', 'condition', 'type'
    ], keep='first')

Update ISTAT codes

In [74]:
# ISTAT codes updated to 2025
df_istat = pd.read_csv('datasets/mun_istat_codes.csv')

# ISTAT codes changes
df_change = pd.read_csv('datasets/changes_istat.csv')

# Uniform ISTAT codes across datasets
add_zeroes(df_istat, ['mun_istat'], 6)
add_zeroes(df_change, ['mun_istat_old', 'mun_istat_new'], 6)

df_istat['mun_istat'] = df_istat['mun_istat'].astype('object')
df_change['mun_istat_old'] = df_change['mun_istat_old'].astype('object')
df_change['mun_istat_new'] = df_change['mun_istat_new'].astype('object')

In [75]:
# Inconsistencies in ISTAT codes (Torre de Busi, Tortoli)
final_df.loc[(final_df['mun_name'] == 'TORRE DE BUSI'), ['mun_istat']] = ['016215']

final_df.loc[final_df['mun_istat'] == '091073', 'mun_istat'] = '114043'

In [110]:
# Update ISTAT codes
updated_df = update_istat(
    df=final_df,
    df_map=df_change, 
    valid_codes=df_istat["mun_istat"], 
    istat_col="mun_istat",
    istat_old = "mun_istat_old",
    istat_new = "mun_istat_new"
)

# Select only suppressed municipalities
supp_df = updated_df[updated_df['suppressed'].isin([True])]

supp_df = supp_df.drop(columns = ['mun_istat', 'mun_istat_updated', 'changed', 'suppressed'])

# Select only non-suppressed municipalities
updated_df = updated_df[updated_df['suppressed'].isin([False])]

updated_df = updated_df.drop(columns = ['mun_istat', 'changed', 'suppressed'])

updated_df = updated_df.rename(columns = {'mun_istat_updated' : 'mun_istat'})

df_istat_new = df_istat[['mun_istat','mun_name']]

In [111]:
# Merge suppressed municipalities with new ISTAT codes (name based)
compare_df = pd.merge(supp_df, df_istat_new, on = 'mun_name', how = 'left')

# Drop municipalities with no match in new ISTAT codes
compare_df = compare_df.dropna(subset = ['mun_istat'])

compare_df['mun_istat'] = compare_df['mun_istat'].astype('str')

# Merge with updated dataset
all_df = pd.concat([updated_df, compare_df], ignore_index=True)

Import region and province names

In [112]:
all_df = pd.merge(all_df, df_istat.drop(columns = ['mun_name', 'prov_istat']), how = 'left', on = 'mun_istat')

Missing/0 values

In [113]:
# Check for missing values
missing_values = all_df.isnull().sum()
print("Missing values in each column:\n", missing_values)

Missing values in each column:
 mun_name             0
year                 0
year_semester        0
semester             0
zone                 0
type                 0
condition        79155
buy_min           1471
buy_max           1471
mun_istat            0
prov_name         8415
reg_name          8415
land_code         8415
dtype: int64


In [114]:
# Deleting missing values
all_df = all_df.dropna()

In [115]:
# Check for 0s in buy columns
zero_values = (all_df == 0).sum()
print("Zero values in each column:\n", zero_values)

Zero values in each column:
 mun_name             0
year                 0
year_semester        0
semester             0
zone                 0
type                 0
condition            0
buy_min          23989
buy_max          23989
mun_istat            0
prov_name            0
reg_name             0
land_code            0
dtype: Int64


In [116]:
# Dropping rows with 0 'buy_min/max' values
all_df = all_df[(all_df['buy_min'] != 0) & (all_df['buy_max'] != 0)]

Correct region and province names

In [117]:
all_df['prov_name'] = all_df['prov_name'].replace({
    'BARLETTAANDRIATRANI' : 'BARLETTA - ANDRIA - TRANI',
    'BOLZANOBOZEN' : 'BOLZANO - BOZEN',
    'FORLICESENA' : 'FORLI - CESENA',
    'MASSACARRARA' : 'MASSA - CARRARA',
    "VALLE D'AOSTAVALLEE D'AOSTE" : "VALLE D'AOSTA",
    'VERBANOCUSIOOSSOLA' : 'VERBANO - CUSIO - OSSOLA'
})

all_df['reg_name'] = all_df['reg_name'].replace({
    'EMILIAROMAGNA' : 'EMILIA ROMAGNA',
    'FRIULIVENEZIA GIULIA' : 'FRIULI VENEZIA GIULIA',
    'TRENTINOALTO ADIGESUDTIROL' : 'TRENTINO ALTO ADIGE - SUDTIROL',
    "VALLE D'AOSTAVALLEE D'AOSTE" : "VALLE D'AOSTA"
})

Correct the problem with Milan old zones

In [118]:
all_df['year'] = all_df['year'].astype(float)

# Delete every zone with 0 as second character for Milan
all_df = all_df[~((all_df['mun_name'] == 'MILANO') & (all_df['year'] >= 2014) & (all_df['zone'].str[1] == '0'))]

all_df['year'] = all_df['year'].astype(int)

Save correspondences of normalised names

In [119]:
names_df = all_df[['mun_istat', 'mun_name', 'prov_name', 'reg_name']]

names_df = names_df.groupby(['mun_istat']).aggregate({
    'mun_name': 'first',
    'prov_name' : 'first',
    'reg_name' : 'first'
}).reset_index()

names_df.to_csv('datasets/names_corr.csv', index = False)

Sobstitute names with non-normalised ones

In [120]:
# Load names correspondences
corr_df = pd.read_csv('datasets/mun_istat_codes_non_normalized.csv')
corr_df = corr_df.drop(columns = ['prov_istat', 'land_code'])
add_zeroes(corr_df, 'mun_istat', 6)

# Drop normalised names
all_df = all_df.drop(columns = ['mun_name', 'reg_name', 'prov_name'])

# Merge with non-normalised names
df_final = pd.merge(all_df, corr_df, on = 'mun_istat', how = 'left')

Save data

In [121]:
df_final.to_csv("datasets/omi_estimate/omi_estimate.csv", index = False)